
### The Dataset is finely curated to cover the edge cases to demonstrate each of the windows function nitty-gritties.

* Ties (`ROW_NUMBER` vs `RANK` vs `DENSE_RANK`): 
    * Notice how plays `1`, `3`, `12`, and `18` all have exactly `1200` seconds watched. When you rank a user's plays by watch time, you will immediately see how `RANK()` leaves gaps in numbering after ties, while `DENSE_RANK()` does not.
* Partition Boundaries (`PARTITION BY user_id`): 
    * With 4 distinct users having varied numbers of watch events (from 4 to 5 plays each), you can test how aggregations and rankings isolate behaviors per user vs. globally across the whole platform.
* Sliding Frames (`ROWS BETWEEN ...`):
    * The chronological played_at timestamps per user allow you to practice building running totals (`SUM(...) OVER (PARTITION BY user_id ORDER BY played_at ...)`) and moving averages.
* Sequential Lookups (`LAG` / `LEAD`):
    * Because timestamps are staggered nicely per user, `LAG(seconds_watched)` will cleanly show what a user watched right before their current session.

In [0]:
-- Create Catalog
CREATE CATALOG IF NOT EXISTS experts;

-- Create Schema inside the experts catalog
CREATE SCHEMA IF NOT EXISTS experts.learn_sql;

In [0]:
-- CREATE Tables inside the experts.learn_sql schema
-- 1. USERS TABLE (4 users with different plans and signup cohorts)
CREATE TABLE IF NOT EXISTS experts.learn_sql.users (user_id BIGINT, signup_date DATE, plan STRING);

INSERT INTO experts.learn_sql.users
  VALUES
    (101, '2025-01-10', 'free'),
    (102, '2025-02-15', 'standard'),
    (103, '2025-03-01', 'premium'),
    (104, '2025-03-10', 'free');

In [0]:
-- 2. TITLES TABLE (6 titles across different genres and release years)
CREATE TABLE IF NOT EXISTS experts.learn_sql.titles (
    title_id BIGINT,
    title_name STRING,
    genre STRING,
    release_year INT
  );

INSERT INTO experts.learn_sql.titles
  VALUES
    (1, 'The Matrix', 'Sci-Fi', 1999),
    (2, 'Inception', 'Sci-Fi', 2010),
    (3, 'The Office', 'Comedy', 2005),
    (4, 'Breaking Bad', 'Drama', 2008),
    (5, 'Interstellar', 'Sci-Fi', 2014),
    (6, 'Friends', 'Comedy', 1994);

In [0]:
-- 3. PLAYS TABLE (22 records designed with ties and chronological sequences)
CREATE TABLE IF NOT EXISTS experts.learn_sql.plays (
    play_id BIGINT,
    user_id BIGINT,
    title_id BIGINT,
    played_at TIMESTAMP,
    seconds_watched INT,
    device STRING,
    country STRING
  );

INSERT INTO experts.learn_sql.plays
  VALUES
    -- User 101 (Free plan, active on mobile & web)
    (1, 101, 1, '2026-03-01 10:00:00', 1200, 'mobile', 'USA'),
    (2, 101, 2, '2026-03-02 14:30:00', 300, 'web', 'USA'),
    (3, 101, 3, '2026-03-04 18:00:00', 1200, 'mobile', 'USA'), -- Tie in seconds_watched with play 1!
    (4, 101, 4, '2026-03-05 20:15:00', 2500, 'web', 'USA'),
    -- User 102 (Standard plan, heavy watcher across tv & mobile)
    (5, 102, 2, '2026-03-01 12:00:00', 3600, 'tv', 'CAN'),
    (6, 102, 5, '2026-03-02 15:00:00', 3600, 'tv', 'CAN'), -- Another tie in seconds_watched!
    (7, 102, 1, '2026-03-03 19:00:00', 1800, 'mobile', 'CAN'),
    (8, 102, 3, '2026-03-04 21:00:00', 1200, 'mobile', 'CAN'),
    (9, 102, 6, '2026-03-05 22:30:00', 900, 'tv', 'CAN'),
    -- User 103 (Premium plan, mix of all devices)
    (10, 103, 5, '2026-03-02 08:30:00', 7200, 'web', 'GBR'),
    (11, 103, 4, '2026-03-03 11:00:00', 2400, 'tv', 'GBR'),
    (12, 103, 1, '2026-03-04 14:00:00', 1200, 'web', 'GBR'), -- Tie with play 1 & 3
    (13, 103, 2, '2026-03-05 16:45:00', 4500, 'mobile', 'GBR'),
    (14, 103, 6, '2026-03-06 19:30:00', 1500, 'tv', 'GBR'),
    -- User 104 (Free plan, short bursts of watch time)
    (15, 104, 3, '2026-03-03 09:00:00', 600, 'mobile', 'AUS'),
    (16, 104, 6, '2026-03-04 13:20:00', 600, 'mobile', 'AUS'), -- Tie with play 15
    (17, 104, 2, '2026-03-05 17:00:00', 1800, 'web', 'AUS'),
    (18, 104, 1, '2026-03-06 21:10:00', 1200, 'web', 'AUS');